# Essence Wars: Training a PPO Agent

This notebook demonstrates training a PPO (Proximal Policy Optimization) agent to play Essence Wars.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yourusername/essence-wars/blob/main/python/notebooks/02_training_ppo.ipynb)

## Setup

In [ ]:
# Install dependencies (uncomment for Colab)
# !pip install essence-wars[train]

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from essence_wars import VectorizedEssenceWars, PyGame
from essence_wars._core import STATE_TENSOR_SIZE, ACTION_SPACE_SIZE

# Force CPU for notebook reliability (GPU optional for larger training)
device = torch.device("cpu")
print(f"Using device: {device}")
print(f"Observation size: {STATE_TENSOR_SIZE}")
print(f"Action space size: {ACTION_SPACE_SIZE}")

## Define the Neural Network

A simple actor-critic network with shared layers and separate policy/value heads.

In [ ]:
class PPONetwork(nn.Module):
    """Actor-Critic network for PPO with action masking."""
    
    def __init__(self, obs_dim: int = STATE_TENSOR_SIZE, 
                 action_dim: int = ACTION_SPACE_SIZE,
                 hidden_dim: int = 128):
        super().__init__()
        
        # Shared feature extractor
        self.shared = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        
        # Policy head (actor)
        self.policy = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
        )
        
        # Value head (critic)
        self.value = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
        )
    
    def forward(self, obs: torch.Tensor, mask: torch.Tensor):
        """Forward pass returning policy logits and value."""
        features = self.shared(obs)
        
        # Get raw logits and mask invalid actions
        logits = self.policy(features)
        # Use large negative value instead of -inf for numerical stability
        logits = torch.where(mask.bool(), logits, torch.tensor(-1e8))
        
        value = self.value(features)
        return logits, value.squeeze(-1)
    
    def get_action(self, obs: torch.Tensor, mask: torch.Tensor, deterministic: bool = False):
        """Sample action from policy."""
        logits, value = self.forward(obs, mask)
        
        if deterministic:
            action = logits.argmax(dim=-1)
        else:
            probs = F.softmax(logits, dim=-1)
            # Clamp for numerical stability
            probs = probs.clamp(min=1e-8)
            probs = probs / probs.sum(dim=-1, keepdim=True)
            action = torch.multinomial(probs, 1).squeeze(-1)
        
        log_prob = F.log_softmax(logits, dim=-1)
        action_log_prob = log_prob.gather(-1, action.unsqueeze(-1)).squeeze(-1)
        
        return action, action_log_prob, value

# Create network
network = PPONetwork().to(device)
print(f"Network parameters: {sum(p.numel() for p in network.parameters()):,}")

## Simple Training Loop

A simplified PPO training loop for demonstration.

In [ ]:
def train_ppo(network, num_envs=8, num_steps=64, total_timesteps=10000, 
              lr=3e-4, gamma=0.99, clip_eps=0.2):
    """Simple PPO training loop."""
    
    optimizer = torch.optim.Adam(network.parameters(), lr=lr)
    env = VectorizedEssenceWars(num_envs=num_envs)
    
    timesteps = 0
    iteration = 0
    rewards_history = []
    
    print(f"Training for {total_timesteps:,} timesteps...")
    
    while timesteps < total_timesteps:
        # Collect rollout
        obs, masks = env.reset(seed=np.random.randint(0, 2**31))
        
        batch_obs, batch_actions, batch_rewards = [], [], []
        batch_log_probs, batch_masks = [], []
        episode_rewards = []
        
        for _ in range(num_steps):
            obs_t = torch.from_numpy(obs).float()
            mask_t = torch.from_numpy(masks).float()
            
            with torch.no_grad():
                actions, log_probs, _ = network.get_action(obs_t, mask_t)
            
            actions_np = actions.numpy().astype(np.uint8)
            next_obs, rewards, dones, next_masks = env.step(actions_np)
            
            batch_obs.append(obs_t)
            batch_actions.append(actions)
            batch_log_probs.append(log_probs)
            batch_rewards.append(torch.from_numpy(rewards).float())
            batch_masks.append(mask_t)
            
            episode_rewards.append(rewards.sum())
            obs, masks = next_obs, next_masks
        
        # Flatten batches
        obs_batch = torch.cat(batch_obs)
        actions_batch = torch.cat(batch_actions)
        old_log_probs = torch.cat(batch_log_probs)
        masks_batch = torch.cat(batch_masks)
        rewards_batch = torch.cat(batch_rewards)
        
        # Simple discounted returns (no GAE for simplicity)
        returns = rewards_batch  # Simplified: just use immediate rewards
        advantages = returns - returns.mean()
        advantages = advantages / (advantages.std() + 1e-8)
        
        # PPO update
        for _ in range(4):  # epochs
            logits, values = network(obs_batch, masks_batch)
            log_probs = F.log_softmax(logits, dim=-1)
            new_log_probs = log_probs.gather(-1, actions_batch.unsqueeze(-1)).squeeze(-1)
            
            # Clipped policy loss
            ratio = torch.exp(new_log_probs - old_log_probs)
            surr1 = ratio * advantages
            surr2 = torch.clamp(ratio, 1-clip_eps, 1+clip_eps) * advantages
            policy_loss = -torch.min(surr1, surr2).mean()
            
            # Value loss
            value_loss = F.mse_loss(values, returns)
            
            # Entropy bonus
            probs = F.softmax(logits, dim=-1).clamp(min=1e-8)
            entropy = -(probs * probs.log()).sum(dim=-1).mean()
            
            loss = policy_loss + 0.5 * value_loss - 0.01 * entropy
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(network.parameters(), 0.5)
            optimizer.step()
        
        timesteps += num_steps * num_envs
        iteration += 1
        rewards_history.append(sum(episode_rewards) / num_envs)
        
        if iteration % 5 == 0:
            mean_reward = np.mean(rewards_history[-5:])
            print(f"Iter {iteration:3d} | Steps: {timesteps:5,} | Loss: {loss.item():.3f} | Reward: {mean_reward:+.2f}")
    
    env.close()
    return rewards_history

print("Training function defined")

## Train the Agent

In [ ]:
# Train (reduced timesteps for notebook demo)
rewards_history = train_ppo(
    network,
    num_envs=8,
    num_steps=32,
    total_timesteps=5000,  # Small for demo; use 100K+ for real training
)
print("\nTraining complete!")

## Evaluate Against Greedy Bot

In [ ]:
def evaluate_agent(network, num_games=20):
    """Evaluate trained agent against greedy bot."""
    network.eval()
    wins = 0
    
    for seed in range(num_games):
        game = PyGame(deck1='artificer_tokens', deck2='broodmother_pack')
        game.reset(seed=seed)
        
        while not game.is_done():
            if game.current_player() == 0:
                obs = torch.from_numpy(game.observe()).float().unsqueeze(0)
                mask = torch.from_numpy(game.action_mask()).float().unsqueeze(0)
                with torch.no_grad():
                    action, _, _ = network.get_action(obs, mask, deterministic=True)
                game.step(action.item())
            else:
                game.step(game.greedy_action())
        
        if game.get_reward(0) > 0:
            wins += 1
    
    network.train()
    return wins / num_games

win_rate = evaluate_agent(network)
print(f"Win rate vs Greedy: {win_rate*100:.1f}%")

## Save Model

In [ ]:
# Save checkpoint
checkpoint = {
    'network_state_dict': network.state_dict(),
    'config': {
        'obs_dim': STATE_TENSOR_SIZE,
        'action_dim': ACTION_SPACE_SIZE,
        'hidden_dim': 128,
    },
}
torch.save(checkpoint, 'ppo_demo.pt')
print("Model saved to ppo_demo.pt")

## Plot Training Progress

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(rewards_history, alpha=0.7)
plt.xlabel('Iteration')
plt.ylabel('Episode Reward')
plt.title('PPO Training Progress')
plt.grid(True, alpha=0.3)
plt.show()

## Next Steps

This was a minimal demo. For serious training:

1. **Increase timesteps**: Use 100K-500K steps
2. **Use GPU**: Change `device = torch.device("cuda")` 
3. **Use the CLI**: `essence-wars train ppo --timesteps 500000`
4. **Add GAE**: Generalized Advantage Estimation improves learning

See [03_evaluation.ipynb](03_evaluation.ipynb) for benchmarking.